In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix
)

data_path = Path("../data/processed/order_level_features.csv")

df = pd.read_csv(
    data_path,
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_estimated_delivery_date"
    ]
)

df.shape

(96456, 37)

In [13]:
train_df = df[
    df["order_purchase_timestamp"] < "2018-01-01"
].copy()

validation_df = df[
    (df["order_purchase_timestamp"] >= "2018-01-01")
    & (df["order_purchase_timestamp"] < "2018-07-01")
].copy()

test_df = df[
    df["order_purchase_timestamp"] >= "2018-07-01"
].copy()

In [14]:
numeric_features = [
    "number_of_items",
    "number_of_unique_products",
    "number_of_unique_sellers",
    "total_product_price",
    "total_freight_value",
    "average_product_weight_g",
    "average_product_length_cm",
    "average_product_height_cm",
    "average_product_width_cm",
    "number_of_product_categories",
    "total_payment_installments",
    "total_payment_value",
    "number_of_payment_records",
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "approval_delay_hours",
    "estimated_delivery_days",
    "estimated_delivery_days_invalid",
    "total_order_cost",
    "freight_ratio",
    "average_item_price",
    "average_product_volume_cm3",
    "product_dimensions_missing",
    "payment_information_missing"
]

categorical_features = [
    "customer_state",
    "primary_payment_type"
]

feature_columns = numeric_features + categorical_features

target_column = "late_delivery"

In [15]:
X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_validation = validation_df[feature_columns].copy()
y_validation = validation_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

In [16]:
def evaluate_model(model, X, y, model_name):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    return pd.DataFrame([{
        "model": model_name,
        "accuracy": round(accuracy_score(y, predictions), 4),
        "precision": round(
            precision_score(y, predictions, zero_division=0),
            4
        ),
        "recall": round(
            recall_score(y, predictions, zero_division=0),
            4
        ),
        "f1_score": round(
            f1_score(y, predictions, zero_division=0),
            4
        ),
        "roc_auc": round(roc_auc_score(y, probabilities), 4),
        "pr_auc": round(
            average_precision_score(y, probabilities),
            4
        )
    }])

In [17]:
numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "one_hot_encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features)
    ]
)

In [18]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](27,)","['number_of_items','number_of_unique_products','number_of_unique_sellers', ...,'payment_information_missing','customer_state','primary_payment_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,27
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedco

In [19]:
logistic_validation_scores = logistic_pipeline.predict_proba(
    X_validation
)[:, 1]

In [20]:
def evaluate_threshold(y_true, risk_scores, threshold):
    predictions = (risk_scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "threshold": threshold,
        "precision": round(
            precision_score(y_true, predictions, zero_division=0),
            4
        ),
        "recall": round(
            recall_score(y_true, predictions, zero_division=0),
            4
        ),
        "f1_score": round(
            f1_score(y_true, predictions, zero_division=0),
            4
        ),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "flagged_orders": int(tp + fp)
    }

In [21]:
thresholds_to_test = [0.20, 0.30, 0.40, 0.50, 0.60]

threshold_results = pd.DataFrame([
    evaluate_threshold(
        y_validation,
        logistic_validation_scores,
        threshold
    )
    for threshold in thresholds_to_test
])

threshold_results

,threshold,precision,recall,f1_score,true_positives,false_positives,false_negatives,true_negatives,flagged_orders
0,0.2,0.1113,0.9364,0.1990,3753,29958,255,6304,33711
1,0.3,0.1344,0.7141,0.2263,2862,18427,1146,17835,21289
2,0.4,0.1791,0.4753,0.2602,1905,8730,2103,27532,10635
3,0.5,0.2177,0.2458,0.2309,985,3540,3023,32722,4525
4,0.6,0.2412,0.0926,0.1338,371,1167,3637,35095,1538


In [22]:
COST_PER_INTERVENTION = 1
COST_OF_MISSED_LATE_ORDER = 5

In [23]:
threshold_results["estimated_business_cost"] = (
    threshold_results["flagged_orders"]
    * COST_PER_INTERVENTION
    + threshold_results["false_negatives"]
    * COST_OF_MISSED_LATE_ORDER
)

threshold_results["cost_per_order"] = (
    threshold_results["estimated_business_cost"]
    / len(y_validation)
).round(4)

threshold_results.sort_values("estimated_business_cost")

,threshold,precision,recall,f1_score,true_positives,false_positives,false_negatives,true_negatives,flagged_orders,estimated_business_cost,cost_per_order
3,0.5,0.2177,0.2458,0.2309,985,3540,3023,32722,4525,19640,0.4877
4,0.6,0.2412,0.0926,0.1338,371,1167,3637,35095,1538,19723,0.4898
2,0.4,0.1791,0.4753,0.2602,1905,8730,2103,27532,10635,21150,0.5252
1,0.3,0.1344,0.7141,0.2263,2862,18427,1146,17835,21289,27019,0.6709
0,0.2,0.1113,0.9364,0.1990,3753,29958,255,6304,33711,34986,0.8688


In [24]:
acceptable_thresholds = threshold_results[
    threshold_results["precision"] >= 0.20
].copy()

acceptable_thresholds.sort_values(
    ["estimated_business_cost", "recall"],
    ascending=[True, False]
)

,threshold,precision,recall,f1_score,true_positives,false_positives,false_negatives,true_negatives,flagged_orders,estimated_business_cost,cost_per_order
3,0.5,0.2177,0.2458,0.2309,985,3540,3023,32722,4525,19640,0.4877
4,0.6,0.2412,0.0926,0.1338,371,1167,3637,35095,1538,19723,0.4898


In [25]:
selected_row = (
    acceptable_thresholds
    .sort_values(
        ["estimated_business_cost", "recall"],
        ascending=[True, False]
    )
    .iloc[0]
)

selected_threshold = selected_row["threshold"]

selected_row

threshold                      0.5000
precision                      0.2177
recall                         0.2458
f1_score                       0.2309
true_positives               985.0000
false_positives             3540.0000
false_negatives             3023.0000
true_negatives             32722.0000
flagged_orders              4525.0000
estimated_business_cost    19640.0000
cost_per_order                 0.4877
Name: 3, dtype: float64

In [26]:
validation_predictions = validation_df[
    [
        "order_id",
        "order_purchase_timestamp",
        "customer_state",
        "late_delivery"
    ]
].copy()

validation_predictions["late_delivery_risk_score"] = (
    logistic_validation_scores
)

validation_predictions["predicted_late_delivery"] = (
    validation_predictions["late_delivery_risk_score"]
    >= selected_threshold
).astype(int)

In [27]:
validation_predictions["risk_level"] = np.select(
    [
        validation_predictions["late_delivery_risk_score"] < 0.30,
        validation_predictions["late_delivery_risk_score"] < 0.60
    ],
    [
        "Low",
        "Medium"
    ],
    default="High"
)

In [28]:
action_map = {
    "Low": "Continue normal order workflow.",
    "Medium": "Monitor seller and carrier progress.",
    "High": "Prioritize seller and carrier follow-up."
}

validation_predictions["recommended_action"] = (
    validation_predictions["risk_level"].map(action_map)
)

In [29]:
validation_predictions.sort_values(
    "late_delivery_risk_score",
    ascending=False
).head(10)

,order_id,order_purchase_timestamp,customer_state,late_delivery,late_delivery_risk_score,predicted_late_delivery,risk_level,recommended_action
93366,0a93b40850d3f4becf2f276666e01340,2018-01-20 14:24:50,RS,1,0.995998,1,High,Prioritize seller and carrier follow-up.
54040,f7923db0430587601c2aef15ec4b8af4,2018-01-20 17:38:58,PR,1,0.992876,1,High,Prioritize seller and carrier follow-up.
69508,daed0f3aefd193de33c31e21b16a3b3a,2018-03-12 15:21:32,MG,1,0.934117,1,High,Prioritize seller and carrier follow-up.
10105,7fc19fa60f9bb0d99b980a47eee3ec28,2018-04-20 20:42:38,AL,0,0.921197,1,High,Prioritize seller and carrier follow-up.
39136,8dbc85d1447242f3b127dda390d56e19,2018-06-22 12:23:19,PB,0,0.916808,1,High,Prioritize seller and carrier follow-up.
65884,4433add7ec1811cbdb34a289d636f1f3,2018-06-12 21:35:49,AL,0,0.908472,1,High,Prioritize seller and carrier follow-up.
86228,46df573bb73f96f2ac4ed61858107f75,2018-03-30 18:55:43,AL,0,0.900218,1,High,Prioritize seller and carrier follow-up.
43182,9eac94dce77597b831e8c0dfc584bc8b,2018-01-18 23:16:44,BA,1,0.898124,1,High,Prioritize seller and carrier follow-up.
59222,eadaca4f30575b6c3a4c4efebce9d252,2018-02-05 20:07:17,AL,1,0.896457,1,High,Prioritize seller and carrier follow-up.
5980,ada302d82697f51b2373ac6da3131588,2018-04-21 18:35:38,AL,1,0.887908,1,High,Prioritize seller and carrier follow-up.


In [34]:
risk_level_summary = (
    validation_predictions
    .groupby("risk_level")
    .agg(
        total_orders=("order_id", "size"),
        actual_late_orders=("late_delivery", "sum"),
        actual_late_rate=("late_delivery", "mean")
    )
    .reset_index()
)

risk_level_summary["actual_late_rate"] = (
    risk_level_summary["actual_late_rate"] * 100
).round(2)

risk_level_summary

,risk_level,total_orders,actual_late_orders,actual_late_rate
0,High,1538,371,24.12
1,Low,18981,1146,6.04
2,Medium,19751,2491,12.61


In [37]:
validation_predictions.to_csv(
    "../data/processed/validation_risk_predictions.csv",
    index=False
)